In [ ]:
import os
from nilearn import datasets

# your target folder
save_dir = "/data/Atlases/Oxford-Cortical-Atlas"   # <-- change this

os.makedirs(save_dir, exist_ok=True)

# Fetch cortical deterministic atlas
atlas_cort = datasets.fetch_atlas_harvard_oxford(
    atlas_name="cort-maxprob-thr50-2mm",
    symmetric_split=False,
    data_dir=save_dir
)

# Fetch subcortical deterministic atlas
atlas_sub = datasets.fetch_atlas_harvard_oxford(
    atlas_name="sub-maxprob-thr50-2mm",
    symmetric_split=False,
    data_dir=save_dir
)

print("Cortical atlas file:", atlas_cort.maps)
print("Cortical labels:", atlas_cort.labels)  # preview first 10 labels

print("Subcortical atlas file:", atlas_sub.maps)
print("Subcortical labels:", atlas_sub.labels)


# Only keep really subcortical labels

In [ ]:
import os
import numpy as np
import nibabel as nib
from pathlib import Path

# ---- INPUTS ----
ATLAS_NII = Path("/data/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-sub-maxprob-thr50-2mm.nii.gz")
# (Optional) try to read an XML labels file next to the atlas; if missing we’ll fall back to nilearn’s labels
ATLAS_XML = ATLAS_NII.with_name("HarvardOxford-Subcortical.xml")

# Keep exactly these labels (Harvard–Oxford naming)
SELECTED = [
    'Left Thalamus', 'Left Caudate', 'Left Putamen', 'Left Pallidum',
    'Brain-Stem',
    'Left Hippocampus', 'Left Amygdala', 'Left Accumbens',
    'Right Thalamus', 'Right Caudate', 'Right Putamen', 'Right Pallidum',
    'Right Hippocampus', 'Right Amygdala', 'Right Accumbens'
]

# ---- OUTPUTS ----
OUT_NII = ATLAS_NII.with_name(ATLAS_NII.name.replace(".nii.gz", "-only_subcortical_regions.nii.gz"))
OUT_LUT = ATLAS_NII.with_name(ATLAS_NII.name.replace(".nii.gz", "-subset_labels.tsv"))

# ---- UTIL: load label names (index -> name) ----
def load_ho_subcortical_labels():
    # 1) try local XML (FSL ships HarvardOxford-Subcortical.xml)
    if ATLAS_XML.exists():
        import xml.etree.ElementTree as ET
        tree = ET.parse(ATLAS_XML)
        root = tree.getroot()
        # FSL XML has elements like <label index="1">Left Thalamus</label>
        labels = {}
        for lab in root.iter("label"):
            idx = int(lab.attrib.get("index", "-1"))
            name = (lab.text or "").strip()
            if idx >= 0:
                labels[idx] = name
        # Ensure background 0 exists
        labels.setdefault(0, "Background")
        # Return dense list from 0..max
        return [labels.get(i, f"Unknown_{i}") for i in range(max(labels.keys()) + 1)]
    # 2) fallback: nilearn (same ordering as their atlas files)
    try:
        from nilearn import datasets
        atlas = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr50-2mm")
        return list(atlas.labels)  # index-aligned; labels[0] == 'Background'
    except Exception:
        raise RuntimeError(
            "Could not find XML next to the atlas and nilearn fallback failed. "
            "Please provide a label list."
        )

# ---- MAIN ----
def main():
    if not ATLAS_NII.exists():
        raise FileNotFoundError(f"Atlas not found: {ATLAS_NII}")

    # load labels and make a name->index map (case/spacing normalized)
    labels = load_ho_subcortical_labels()  # list indexed by integer voxel value
    norm = lambda s: s.lower().replace(" ", "").replace("-", "")
    name_to_idx = {norm(n): i for i, n in enumerate(labels)}

    wanted_idx = []
    missing = []
    for name in SELECTED:
        idx = name_to_idx.get(norm(name))
        if idx is None:
            missing.append(name)
        else:
            wanted_idx.append(idx)

    if missing:
        print("[WARN] The following requested labels were not found in the atlas:")
        for m in missing:
            print("   -", m)
        # continue with those we found
    wanted_idx = sorted(set(wanted_idx))
    if not wanted_idx:
        raise ValueError("None of the requested labels were found. Aborting.")

    # load atlas image and data
    img = nib.load(str(ATLAS_NII))
    data = np.asanyarray(img.dataobj)  # int labels (0..N)
    out = np.zeros_like(data, dtype=np.int16)

    # relabel selected indices to 1..K in the order we defined above
    idx_to_new = {old: new for new, old in enumerate(wanted_idx, start=1)}
    for old_idx, new_idx in idx_to_new.items():
        out[data == old_idx] = new_idx

    # save NIfTI with same affine / header (but ensure integer dtype)
    out_img = nib.Nifti1Image(out, affine=img.affine, header=img.header.copy())
    out_img.set_data_dtype(np.int16)
    nib.save(out_img, str(OUT_NII))
    print(f"[SAVED] subset atlas: {OUT_NII}")

    # write label table TSV
    with open(OUT_LUT, "w", encoding="utf-8") as f:
        f.write("index\tlabel\n")
        for old_idx in wanted_idx:
            f.write(f"{idx_to_new[old_idx]}\t{labels[old_idx]}\n")
    print(f"[SAVED] labels: {OUT_LUT}")

    # quick summary
    print(f"Kept {len(wanted_idx)} labels → indices 1..{len(wanted_idx)}")
    print("First few:", [labels[i] for i in wanted_idx[:5]])

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path

# ---- INPUT ----
ATLAS_NII = Path("/data/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-cortl-maxprob-thr50-2mm.nii.gz")
ATLAS_XML = ATLAS_NII.with_name("HarvardOxford-Cortical.xml")  # if present, we’ll read labels from here

# ---- OUTPUT ----
OUT_NII = ATLAS_NII.with_name(ATLAS_NII.name.replace(".nii.gz", "-without_background.nii.gz"))
OUT_LUT = ATLAS_NII.with_name(ATLAS_NII.name.replace(".nii.gz", "-nobg_labels.tsv"))

# If you want to *reindex* labels to 1..K (drop background=0 and compress),
# set this to True. By default we keep the original label indices.
REINDEX_TO_COMPACT = False

def load_ho_cortical_labels():
    """Return list of labels (index-aligned), labels[0] should be 'Background'."""
    # 1) Try local FSL XML
    if ATLAS_XML.exists():
        import xml.etree.ElementTree as ET
        tree = ET.parse(ATLAS_XML)
        root = tree.getroot()
        labels = {}
        for lab in root.iter("label"):  # <label index="1">Frontal Pole</label>
            idx = int(lab.attrib.get("index", "-1"))
            name = (lab.text or "").strip()
            if idx >= 0:
                labels[idx] = name
        labels.setdefault(0, "Background")
        return [labels.get(i, f"Unknown_{i}") for i in range(max(labels.keys()) + 1)]
    # 2) Fallback: nilearn dataset
    try:
        from nilearn import datasets
        print("ERROR: Could not find label XML next to the atlas. Falling back to nilearn’s labels.")
        atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr50-2mm")
        return list(atlas.labels)  # labels[0] == 'Background'
    except Exception:
        raise RuntimeError("Could not find label XML and nilearn fallback failed. Provide a label list.")

def main():
    if not ATLAS_NII.exists():
        raise FileNotFoundError(f"Atlas not found: {ATLAS_NII}")

    # Load atlas and labels
    img = nib.load(str(ATLAS_NII))
    data = np.asanyarray(img.dataobj)  # integer labels: 0..N
    labels = load_ho_cortical_labels()
    if not labels or labels[0].lower() != "background":
        print("[WARN] labels[0] is not 'Background' as expected. Continuing.")

    if REINDEX_TO_COMPACT:
        # Create a compact atlas: map unique labels >0 to 1..K, background stays 0
        uniq = np.unique(data)
        uniq = uniq[uniq > 0]
        idx_to_new = {old: new for new, old in enumerate(uniq, start=1)}
        out = np.zeros_like(data, dtype=np.int16)
        for old, new in idx_to_new.items():
            out[data == old] = new
        # Save NIfTI
        out_img = nib.Nifti1Image(out, affine=img.affine, header=img.header.copy())
        out_img.set_data_dtype(np.int16)
        nib.save(out_img, str(OUT_NII))
        print(f"[SAVED] cortical atlas (compact, no background label in LUT): {OUT_NII}")

        # Save LUT without background, remapped to 1..K
        with open(OUT_LUT, "w", encoding="utf-8") as f:
            f.write("index\tlabel\n")
            for old in uniq:
                new = idx_to_new[old]
                name = labels[old] if old < len(labels) else f"Label_{old}"
                f.write(f"{new}\t{name}\n")
        print(f"[SAVED] labels TSV (no background): {OUT_LUT}")

    else:
        # Keep original indices/data; just write a LUT without the background row
        nib.save(nib.Nifti1Image(data.astype(np.int16), img.affine, img.header), str(OUT_NII))
        print(f"[SAVED] cortical atlas copy (data unchanged): {OUT_NII}")

        with open(OUT_LUT, "w", encoding="utf-8") as f:
            f.write("index\tlabel\n")
            for i, name in enumerate(labels):
                if i == 0:   # skip background
                    continue
                f.write(f"{i}\t{name}\n")
        print(f"[SAVED] labels TSV (no background row): {OUT_LUT}")

if __name__ == "__main__":
    main()
